# SpectraShift Week 6: freeze the ImageNet RGB contract
Use CPU with Internet off. Attach `spectrashift-source-v5`, `spectrashift-week2-frozen`, `spectrashift-week5-contracts`, and `spectrashift-week5-complete`.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week6.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 6 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week6-source')
    if source_work.exists():
        shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 6 source tree found'
projects.sort(key=lambda path: (0 if 'spectrashift-source' in str(path) else 1, len(str(path))))
PROJECT = projects[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        by_hash.setdefault(digest, path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

WORK = Path('/kaggle/working/spectrashift-week6-contracts')
WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = unique_file('partitions.parquet')
NORMALIZATION = unique_file('normalization.json')
FREEZE = unique_file('freeze_summary.json')
WEEK5_CONTRACTS = unique_file('week5_contracts_summary.json')
WEEK5_SUMMARY = unique_file('week5_run_summary.json')
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
config = yaml.safe_load((PROJECT / 'configs/downstream/week6.yaml').read_text())
config['data']['manifest_path'] = str(MANIFEST)
config['data']['staged_root'] = str(STAGED)
config['data']['normalization_path'] = str(NORMALIZATION)
config['data']['freeze_summary_path'] = str(FREEZE)
config['contracts']['output_dir'] = str(WORK)
config['contracts']['week5_contracts_summary_path'] = str(WEEK5_CONTRACTS)
config['contracts']['week5_summary_path'] = str(WEEK5_SUMMARY)
RUNTIME_CONFIG = WORK / 'week6.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print({'project': str(PROJECT), 'work': str(WORK)})


In [ ]:
from spectrashift.train.week6 import freeze_week6_contracts

summary = freeze_week6_contracts(RUNTIME_CONFIG)
print(json.dumps(summary, indent=2))
assert summary['week6_contracts_complete']
assert summary['rgb_band_order'] == ['B04', 'B03', 'B02']
assert summary['rgb_fit_partition'] == 'U' and summary['rgb_fit_patch_count'] == 20000
assert summary['evaluation_labels_loaded'] is False
